# `mutate` — Reference

`mutate` creates or overwrites columns from a `qry()`-style spec string, each entry evaluated in order via pandas `eval()` — a plain formula per column, no lambda required.

| Syntax | Meaning |
|---|---|
| `"new_col: expr"` | one derived column |
| `"a: expr1, b: expr2"` | several in one call (comma-separated) |
| `"'new_col': expr"` | quoting the key is optional, same as `qry()` |
| `"new_col: if_else(cond, true_val, false_val)"` | dplyr-style two-branch conditional |
| `"new_col: case_when(cond1: v1, cond2: v2, True: default)"` | dplyr-style multi-branch conditional |

Column names *inside* the expression must stay unquoted — see the example below.

---

In [44]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt

penguins = pt.sample_data['penguins']

## A single derived column

In [45]:
# Body mass index style ratio — a plain arithmetic formula, no lambda needed
(penguins
 .mutate('bmi: body_mass_g / bill_length_mm ** 2')
 .select('species', 'body_mass_g', 'bill_length_mm', 'bmi')
 .sample(10)
)

,species,body_mass_g,bill_length_mm,bmi
113,Adelie,4275.0,42.2,2.400553
239,Gentoo,5350.0,48.7,2.255775
103,Adelie,4250.0,37.8,2.974441
168,Chinstrap,3300.0,50.3,1.304301
23,Adelie,3950.0,38.2,2.706889
270,Gentoo,4850.0,46.6,2.233417
309,Gentoo,5550.0,52.1,2.044643
299,Gentoo,5950.0,45.2,2.912327
114,Adelie,3900.0,39.6,2.486991
249,Gentoo,5550.0,50.0,2.220000


In [46]:
# Column names inside the expression must stay unquoted — quoting one turns it
# into a string literal, not a column reference, and breaks the arithmetic
try:
    penguins.mutate("bmi: 'body_mass_g' / 'bill_length_mm' ** 2")
except TypeError as e:
    print('TypeError:', e)

TypeError: unsupported operand type(s) for ** or pow(): 'str' and 'int'


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [47]:
# Two independent derived columns in one call
(penguins
 .mutate('heavy: body_mass_g > 4000, mass_kg: body_mass_g / 1000')
 .select('species', 'body_mass_g', 'mass_kg', 'heavy')
 .sample(10)
)

,species,body_mass_g,mass_kg,heavy
22,Adelie,3800.0,3.800,False
0,Adelie,3750.0,3.750,False
316,Gentoo,4925.0,4.925,True
45,Adelie,4600.0,4.600,True
157,Chinstrap,3950.0,3.950,False
322,Gentoo,4975.0,4.975,True
30,Adelie,3250.0,3.250,False
266,Gentoo,4200.0,4.200,True
218,Chinstrap,4100.0,4.100,True
63,Adelie,4050.0,4.050,True


In [48]:
# mass_lb references mass_kg, derived by the entry just before it
(penguins
 .mutate('mass_kg: body_mass_g / 1000, mass_lb: mass_kg * 2.20462')
 .select('species', 'mass_kg', 'mass_lb')
 .sample(10)
)

,species,mass_kg,mass_lb
38,Adelie,3.300,7.275246
229,Gentoo,5.150,11.353793
47,Adelie,2.975,6.558744
116,Adelie,2.900,6.393398
201,Chinstrap,3.675,8.101978
142,Adelie,3.050,6.724091
247,Gentoo,5.650,12.456103
171,Chinstrap,4.400,9.700328
285,Gentoo,5.700,12.566334
78,Adelie,3.550,7.826401


## String comparisons, and optional key quoting
Quoting the key (`'is_adelie'` vs `is_adelie`) is optional, same as `qry()`. String literals *inside* the expression (e.g. `'Adelie'`) still need real quotes — only the column names must stay bare.

In [49]:
unquoted = penguins.mutate("is_adelie: species == 'Adelie'")
quoted = penguins.mutate("'is_adelie': species == 'Adelie'")
(unquoted['is_adelie'] == quoted['is_adelie']).all()

np.True_

## Overwriting an existing column

In [50]:
# mutate() can overwrite a column in place, e.g. converting units
(penguins
 .mutate('body_mass_g: body_mass_g / 1000')
 .select('species', 'body_mass_g')
 .sample(10)
)

,species,body_mass_g
56,Adelie,3.550
130,Adelie,3.325
327,Gentoo,5.500
176,Chinstrap,3.300
192,Chinstrap,3.950
81,Adelie,4.700
173,Chinstrap,3.400
292,Gentoo,5.100
174,Chinstrap,2.900
277,Gentoo,5.000


## Column names with spaces — backtick quoting
`pandas.eval()` uses **backticks**, not the single/double quotes used elsewhere in pytae, to reference a column name containing a space.

In [51]:
import pandas as pd
spaced = pd.DataFrame({'body mass g': [3000.0, 4000.0], 'bill length mm': [30.0, 40.0]})
spaced.mutate('bmi: `body mass g` / `bill length mm` ** 2')

,body mass g,bill length mm,bmi
0,3000.0,30.0,3.333333
1,4000.0,40.0,2.500000


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `qry()`.

In [52]:
(penguins
 .mutate('bmi: body_mass_g / bill_length_mm ** 2')
 .qry({'bmi': ('>', 2)})
 .select('species', 'bmi')
 .sample(10)
)

,species,bmi
250,Gentoo,2.346589
233,Gentoo,2.497268
69,Adelie,2.546874
266,Gentoo,2.028740
276,Gentoo,2.241404
117,Adelie,2.713309
60,Adelie,2.471577
228,Gentoo,2.346804
251,Gentoo,2.565726
123,Adelie,2.260846


## Conditional column creation — `if_else()` and `case_when()`
Plain `eval()` has no ternary/`where()` support, but `mutate()` recognizes two dplyr-style expression forms by name and evaluates them via `np.where()`/`np.select()` instead: `if_else(condition, true_value, false_value)` and `case_when(cond1: val1, cond2: val2, ..., True: default)`. Conditions/non-string values are still `eval()` expressions; string outcomes need quotes.

In [ ]:
# if_else(condition, true_value, false_value) — like dplyr's if_else()
(penguins
 .mutate("weight_class: if_else(body_mass_g > 4000, 'heavy', 'light')")
 .select('species', 'body_mass_g', 'weight_class')
 .sample(10)
)

,species,body_mass_g,bonus
192,Chinstrap,3950.0,10
4,Adelie,3450.0,10
234,Gentoo,4200.0,100
276,Gentoo,4300.0,100
11,Adelie,3700.0,10
307,Gentoo,5300.0,100
267,Gentoo,5400.0,100
136,Adelie,3175.0,10
138,Adelie,3400.0,10
173,Chinstrap,3400.0,10


In [ ]:
# case_when(cond1: val1, cond2: val2, ..., True: default) — like dplyr's case_when()
# checked in order, first match wins; `True` is an optional catch-all default and must be listed last
(penguins
 .mutate("size_class: case_when(body_mass_g >= 4500: 'large', body_mass_g >= 3500: 'medium', True: 'small')")
 .select('species', 'body_mass_g', 'size_class')
 .sample(10)
)

,species,body_mass_g,weight_class
231,Gentoo,5550.0,heavy
165,Chinstrap,4050.0,heavy
16,Adelie,3450.0,light
123,Adelie,3875.0,light
317,Gentoo,4875.0,heavy
300,Gentoo,4625.0,heavy
202,Chinstrap,3325.0,light
240,Gentoo,5700.0,heavy
302,Gentoo,4725.0,heavy
249,Gentoo,5550.0,heavy
